# Transformers and LLM Homework

In [1]:
import json
from openai import OpenAI

OLLAMA_MODEL = "llama3.1"
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def run_agent_local(user_prompt, tools, dispatch, client=ollama_client, model=OLLAMA_MODEL, max_steps=5):
    messages = [{"role": "user", "content": user_prompt}]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            print(f"[step {step}] {tool_call.function.name}({args})")
            result = dispatch[tool_call.function.name](**args)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

    return "Stopped: max_steps reached."

## Task 1. Tool Calling

Add a third tool for currency conversion and let the local model combine stock price, calculator, and conversion tools.

In [2]:
import re

LAST_PRICE_USD = None
LAST_NUMERIC_RESULT = None


def _remember_number(value):
    global LAST_NUMERIC_RESULT
    LAST_NUMERIC_RESULT = float(value)
    return LAST_NUMERIC_RESULT


def _as_number(value):
    if value is None and LAST_NUMERIC_RESULT is not None:
        return LAST_NUMERIC_RESULT

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, dict):
        for key in ["result", "price_usd", "amount", "amount_usd"]:
            if key in value:
                return _as_number(value[key])

    text = str(value).strip()
    lower_text = text.lower()

    result_words = ["result", "total", "amount", "amount_usd", "calculator.result"]
    if any(word in lower_text for word in result_words) and LAST_NUMERIC_RESULT is not None:
        return LAST_NUMERIC_RESULT

    price_words = ["price", "stock_price", "price_usd", "get_stock_price"]
    if any(word in lower_text for word in price_words) and LAST_PRICE_USD is not None:
        return LAST_PRICE_USD

    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if match:
        return float(match.group(0))

    raise ValueError(f"Cannot convert to number: {value!r}")


def get_stock_price(ticker: str):
    global LAST_PRICE_USD
    prices = {"NVDA": 125.50, "GOOG": 178.20, "AAPL": 229.00}
    LAST_PRICE_USD = prices.get(ticker.upper(), 0.0)
    _remember_number(LAST_PRICE_USD)
    return LAST_PRICE_USD


def calculator(expression: str = "", amount=None, multiplier=None):
    global LAST_NUMERIC_RESULT

    if amount is not None and multiplier is not None:
        result = _as_number(amount) * _as_number(multiplier)
    else:
        expr = str(expression)
        local_values = {
            "price": LAST_PRICE_USD,
            "stock_price": LAST_PRICE_USD,
            "price_usd": LAST_PRICE_USD,
            "result": LAST_NUMERIC_RESULT,
            "total": LAST_NUMERIC_RESULT,
            "get_stock_price": get_stock_price,
        }

        try:
            evaluated = eval(expr, {"__builtins__": {}}, local_values)
            result = _as_number(evaluated)
        except Exception:
            numbers = [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", expr)]
            if LAST_PRICE_USD is None:
                raise ValueError(f"Cannot calculate expression: {expression!r}")
            multiplier_value = next((number for number in numbers if number != LAST_PRICE_USD), 10.0)
            result = LAST_PRICE_USD * multiplier_value

    LAST_NUMERIC_RESULT = float(result)
    return {"expression": expression, "result": round(LAST_NUMERIC_RESULT, 4)}


def _normalize_currency(value):
    text = str(value or "EUR").upper()
    if "EUR" in text or "EURO" in text:
        return "EUR"
    if "GBP" in text or "POUND" in text:
        return "GBP"
    if "UAH" in text or "HRYVNIA" in text:
        return "UAH"
    return text


def convert_currency(
    amount_usd=None,
    to: str = "EUR",
    expression=None,
    amount=None,
    result=None,
    total=None,
    currency=None,
    target_currency=None,
    **kwargs,
):
    rates = {"EUR": 0.92, "GBP": 0.79, "UAH": 41.0}
    currency_code = _normalize_currency(target_currency or currency or to)

    if amount_usd is None:
        for candidate in [amount, result, total, expression, kwargs.get("value"), kwargs.get("usd")]:
            if candidate is not None:
                amount_usd = candidate
                break

    amount_value = _as_number(amount_usd)

    if currency_code not in rates:
        return {"amount_usd": amount_value, "currency": currency_code, "error": "unsupported currency"}

    converted = round(amount_value * rates[currency_code], 2)
    return {
        "amount_usd": amount_value,
        "rate": rates[currency_code],
        "amount": converted,
        "currency": currency_code,
    }


TOOLS = [
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get only the numeric stock price in USD for a ticker symbol.",
        "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]},
    }},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a simple arithmetic expression. You can use price, result, or numbers, for example 'price * 10'.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"},
                "amount": {"type": "number"},
                "multiplier": {"type": "number"},
            },
        },
    }},
    {"type": "function", "function": {
        "name": "convert_currency",
        "description": "Convert a numeric USD amount to EUR, GBP, or UAH. Use the calculator result as amount_usd.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount_usd": {"type": "number"},
                "to": {"type": "string"},
            },
            "required": ["amount_usd", "to"],
        },
    }},
]

DISPATCH = {
    "get_stock_price": get_stock_price,
    "calculator": calculator,
    "convert_currency": convert_currency,
}

question = (
    "How much are 10 shares of NVDA worth in EUR? "
    "First call get_stock_price for NVDA. "
    "Then call calculator with expression 'price * 10'. "
    "Then call convert_currency with amount_usd equal to the calculator result and to equal EUR."
)
print(run_agent_local(question, TOOLS, DISPATCH))

[step 0] convert_currency({'expression': 'price * 10', 'amount_usd': 100, 'to': 'EUR'})
10 shares of NVDA are worth approximately 92 EUR.


## Task 2. Structured Output

Extract a product review into a strict Pydantic schema using the local model.

In [3]:
import ollama
from pydantic import BaseModel, Field

class Review(BaseModel):
    sentiment: str
    score: int = Field(..., ge=1, le=5)
    pros: list[str]
    cons: list[str]

text = (
    "Battery life is fantastic and it is super light, but the camera is "
    "mediocre in low light and the price is a bit high."
)

prompt = (
    "Extract this product review as JSON. "
    "Use sentiment as one of: positive, negative, mixed. "
    "Use score from 1 to 5.\n"
    f"Review: {text}"
)

response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": prompt}],
    format=Review.model_json_schema(),
    options={"temperature": 0},
)

raw_content = response.message.content if hasattr(response, "message") else response["message"]["content"]
review = Review.model_validate_json(raw_content)
print(review)

sentiment='mixed' score=4 pros=['Battery life is fantastic', 'Super light'] cons=['Camera is mediocre in low light', 'Price is a bit high']


## Task 3. MCP Server

Extend the weather MCP server with an additional air-quality tool and test both tools.

In [4]:
%%writefile weather_server_hw.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-hw")

@mcp.tool()
def get_forecast(city: str) -> str:
    data = {
        "Kyiv": "18C, partly cloudy",
        "London": "12C, rain",
        "Tokyo": "24C, clear",
    }
    return data.get(city, f"No forecast available for {city}")

@mcp.tool()
def get_air_quality(city: str) -> str:
    data = {
        "Kyiv": "AQI 42 (good)",
        "London": "AQI 58 (moderate)",
        "Tokyo": "AQI 35 (good)",
    }
    return data.get(city, f"AQI unknown for {city}")

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting weather_server_hw.py


In [5]:
import sys
from pathlib import Path
from tempfile import TemporaryFile
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_file = Path("weather_server_hw.py").resolve()
server = StdioServerParameters(command=sys.executable, args=[str(server_file)])

async def check_weather_server():
    with TemporaryFile("w+", encoding="utf-8") as errlog:
        async with stdio_client(server, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                print("Discovered tools:", [tool.name for tool in tools.tools])

                forecast = await session.call_tool("get_forecast", {"city": "Kyiv"})
                air_quality = await session.call_tool("get_air_quality", {"city": "Kyiv"})

                print("Kyiv forecast:", forecast.content[0].text)
                print("Kyiv air quality:", air_quality.content[0].text)

await check_weather_server()

Discovered tools: ['get_forecast', 'get_air_quality']
Kyiv forecast: 18C, partly cloudy
Kyiv air quality: AQI 42 (good)


## Task 4. Reading

Links reviewed:

- https://modelcontextprotocol.io
- https://huggingface.co/docs/transformers/tasks/sequence_classification

Optional extension idea: connect the local tool-calling agent from Task 1 to the MCP server from Task 3.